# Skin Condition Detector — HAM10000 Multi-Class Training (Kaggle)

Train a 7-class skin condition classifier using transfer learning on ResNet18.

**Before running:**
1. **Add HAM10000** — Add Input → **kmader/skin-cancer-mnist-ham10000**
2. **Enable GPU** — Settings → **Accelerator** → **GPU T4 x2**
3. Run all cells

**Output:** `skin_model.pth` — download and place in `models/skin_model.pth`

In [ ]:
import os
import copy
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

CLASS_CODES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CODE_TO_IDX = {code: i for i, code in enumerate(CLASS_CODES)}
BATCH_SIZE = 32
NUM_EPOCHS_STAGE1 = 10
NUM_EPOCHS_STAGE2 = 20
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

INPUT_DIR = Path('/kaggle/input')
metadata_candidates = list(INPUT_DIR.rglob('HAM10000_metadata.csv'))
if not metadata_candidates:
    raise FileNotFoundError('Add HAM10000 dataset as input.')
METADATA_PATH = metadata_candidates[0]
DATA_DIR = METADATA_PATH.parent
print(f'DATA_DIR: {DATA_DIR}')

In [ ]:
df = pd.read_csv(METADATA_PATH)
df = df[df['dx'].isin(CLASS_CODES)].reset_index(drop=True)
df['label'] = df['dx'].map(CODE_TO_IDX)

image_id_to_path = {
    os.path.splitext(os.path.basename(path))[0]: path
    for path in glob(str(DATA_DIR / '**' / '*.jpg'), recursive=True)
}
df['image_path'] = df['image_id'].map(image_id_to_path)
df = df.dropna(subset=['image_path']).reset_index(drop=True)

print(f'Images: {len(df)}')
print(df['dx'].value_counts())

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class SkinDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        return self.transform(image), row['label']

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=RANDOM_SEED)
train_loader = DataLoader(SkinDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(SkinDataset(val_df, val_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
def build_model(num_classes=7):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_classes))
    return model

def train_epochs(model, loader, criterion, optimizer, epochs):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f'Epoch {epoch+1}/{epochs} — loss: {running_loss/len(loader):.4f}')

model = build_model().to(DEVICE)
criterion = nn.CrossEntropyLoss()

# Stage 1: freeze backbone
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
train_epochs(model, train_loader, criterion, optimizer, NUM_EPOCHS_STAGE1)

# Stage 2: fine-tune all layers
for param in model.parameters():
    param.requires_grad = True
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
train_epochs(model, train_loader, criterion, optimizer, NUM_EPOCHS_STAGE2)

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        preds = model(images).argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=CLASS_CODES))

In [ ]:
OUTPUT_PATH = '/kaggle/working/skin_model.pth'
checkpoint = {
    'model_state_dict': model.state_dict(),
    'class_codes': CLASS_CODES,
}
torch.save(checkpoint, OUTPUT_PATH)
print(f'Saved to {OUTPUT_PATH} — download from Output tab')